In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

val_df = pd.read_csv(
    "../data/processed/validation_processed.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_processed.csv"
)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (68976, 3)
Validation: (8878, 3)
Test: (9172, 3)


,labels,text,clean_text
0,pt,"os chefes de defesa da estónia, letónia, lituâ...","os chefes de defesa da estónia, letónia, lituâ..."
1,bg,размерът на хоризонталната мрежа може да бъде ...,размерът на хоризонталната мрежа може да бъде ...
2,zh,很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把...,很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把...
3,th,สำหรับ ของเก่า ที่ จริงจัง ลอง honeychurch ...,สำหรับ ของเก่า ที่ จริงจัง ลอง honeychurch ของ...
4,ru,Он увеличил давление .,Он увеличил давление .


In [6]:
X_train = train_df["clean_text"]
y_train = train_df["labels"]

X_val = val_df["clean_text"]
y_val = val_df["labels"]

X_test = test_df["clean_text"]
y_test = test_df["labels"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVí dụ text:")
print(X_train.iloc[0])

print("\nLabel:")
print(y_train.iloc[0])

X_train: (68976,)
y_train: (68976,)

Ví dụ text:
os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.

Label:
pt


In [7]:
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    lowercase=True,
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

In [9]:
X_train_tfidf = tfidf_vectorizer.fit_transform(
    X_train
)
print(
    "TF-IDF train shape:",
    X_train_tfidf.shape
)

TF-IDF train shape: (68976, 100000)


In [11]:
X_val_tfidf = tfidf_vectorizer.transform(
    X_val
)

X_test_tfidf = tfidf_vectorizer.transform(
    X_test
)
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)

Train: (68976, 100000)
Validation: (8878, 100000)
Test: (9172, 100000)


In [14]:
print(
    "Số lượng đặc trưng:",
    len(tfidf_vectorizer.vocabulary_)
)
feature_names = tfidf_vectorizer.get_feature_names_out()

print(feature_names[:50])

Số lượng đặc trưng: 100000
[' !' ' ! ' ' ! j' ' ! l' ' !!' ' !! ' ' !!!' ' !!! ' ' !!!!' ' "' ' " '
 ' " "' ' " и' ' " н' ' " п' ' " с' ' " ا' ' "d' ' "n' ' "p' ' "s' ' $'
 ' $ ' ' $1' ' %' ' % ' ' &' ' & ' ' & p' ' & po' " '" " ' " " ' '"
 " ' ' " " ' a" " ' a " " ' d" " ' da" " ' de" " ' do" " ' e" " ' e "
 " ' i" " ' i " " ' in" " ' l" " ' m" " ' m " " ' n" " ' nd"]


In [15]:
sample_index = 0

sample_vector = X_train_tfidf[
    sample_index
]

print("Text:")
print(X_train.iloc[sample_index])

print("\nLanguage:")
print(y_train.iloc[sample_index])

print("\nVector shape:")
print(sample_vector.shape)

print("\nSố feature khác 0:")
print(sample_vector.nnz)

Text:
os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.

Language:
pt

Vector shape:
(1, 100000)

Số feature khác 0:
359


In [16]:
sample_vector = X_train_tfidf[0]

indices = sample_vector.indices
values = sample_vector.data

sample_features = pd.DataFrame({
    "ngram": feature_names[indices],
    "tfidf": values
})

sample_features = sample_features.sort_values(
    by="tfidf",
    ascending=False
)

sample_features.head(20)

,ngram,tfidf
324,"nia,",0.140493
227,"nia,",0.140493
228,"ia,",0.136885
112,"ia,",0.136514
110,tón,0.110711
113,"a,",0.107844
20,"a,",0.107399
211,efes,0.106960
241,anha,0.106779
229,"a, l",0.104467


In [17]:
total_values = (
    X_train_tfidf.shape[0]
    * X_train_tfidf.shape[1]
)

non_zero_values = X_train_tfidf.nnz

sparsity = (
    1
    - non_zero_values / total_values
) * 100

print(
    f"Sparsity: {sparsity:.2f}%"
)

Sparsity: 99.74%


In [18]:
os.makedirs(
    "../models",
    exist_ok=True
)

In [19]:
joblib.dump(
    tfidf_vectorizer,
    "../models/tfidf_vectorizer.joblib"
)

print("TF-IDF vectorizer saved!")

TF-IDF vectorizer saved!


In [20]:
loaded_vectorizer = joblib.load(
    "../models/tfidf_vectorizer.joblib"
)

print(
    "Number of features:",
    len(
        loaded_vectorizer
        .get_feature_names_out()
    )
)

Number of features: 100000


In [21]:
sample_text = [
    "Bonjour, comment allez-vous aujourd'hui?"
]

sample_tfidf = loaded_vectorizer.transform(
    sample_text
)

print("Shape:", sample_tfidf.shape)
print(
    "Non-zero features:",
    sample_tfidf.nnz
)

Shape: (1, 100000)
Non-zero features: 100


## Kết luận

- Dữ liệu văn bản sau tiền xử lý được sử dụng làm đầu vào cho quá trình trích xuất đặc trưng.
- Phương pháp TF-IDF kết hợp Character N-gram được sử dụng để biểu diễn văn bản dưới dạng vector số.
- Character N-gram từ 2 đến 5 ký tự được lựa chọn nhằm khai thác các mẫu ký tự đặc trưng của từng ngôn ngữ.
- TF-IDF Vectorizer chỉ được huấn luyện trên tập train nhằm tránh hiện tượng rò rỉ dữ liệu từ tập validation và test.
- Tập validation và test được biến đổi bằng cùng Vectorizer đã học từ tập train.
- Ma trận đặc trưng được lưu dưới dạng sparse matrix, phù hợp với dữ liệu văn bản có số lượng đặc trưng lớn.
- TF-IDF Vectorizer đã được lưu lại để sử dụng trong quá trình huấn luyện mô hình và triển khai hệ thống sau này.